# Gap 2 on Colab: 1024 atoms, standard and fast cooling

This notebook owns **only** `ka2d_1024_standard` and `ka2d_1024_fast`. Run it instead of PSC for these conditions; keep one worker per condition. The four Kaggle conditions are already complete. No new model training is required.

To conserve your **100 compute units**, use three phases in the **same saved notebook and Drive folder**:

| Phase | Set `PHASE` | Runtime hardware | Work |
|---|---|---|---|
| 1 | `prepare` | CPU / accelerator None | 100 independent hosts and 256 matched requests per condition |
| 2 | `sample` | One GPU, initially T4 if available | Frozen seed0, seed1, seed2 and same-checkpoint seed0 sham |
| 3 | `relax` | CPU / accelerator None | Original CPU FIRE relaxation, matched insertion, validation and combined report |

For each phase select **Runtime → Change runtime type**, set `PHASE` below, then **Run all**. Save a copy of this notebook to Drive so configuration edits persist. Changing the runtime deletes local `/content` files; checkpoints are restored from **your mounted Google Drive**, with GitHub as an additional published copy. Add the **GITHUB_TOKEN** secret in Colab's key panel, enable notebook access, and grant repository **Contents: Read and write** for `chilalaabubakar/IDL-PROJECT`. Kaggle secrets do not automatically appear in Colab.

**100 units is not 100 GPU-hours.** Unit rates vary by assigned accelerator and account. This split avoids holding a paid GPU during the long CPU relaxation. It does not reduce total CPU work, and free CPU sessions have their own availability/time limits. Full Colab runtime and unit consumption have not been measured. A four-hour GPU session cap is a time guard, not a guarantee of staying within a particular unit balance. If desired, enter your current CU/hour rate below; the first saved GPU batches provide an updated estimate. The notebook never changes billing or enables purchases.

Each CPU session has a 10.5-hour budget; each GPU session has a four-hour budget. Paused phases resume with the same settings. Valid checkpoints are saved every five minutes and at stage boundaries. The GPU phase automatically disconnects after its execution block (including a planned pause) to avoid continuing GPU allocation. Inspect saved logs/`progress.json` on Drive after a failure, correct the issue, and rerun the same phase. You can set `AUTO_DISCONNECT_GPU=False` while debugging; then disconnect the runtime manually.

The protocol is unchanged: frozen checkpoints/thresholds/training patch library, guidance 2, radius 6, 100+100 sampling steps, batch 4, seeds `1000+batch_start`, double-precision CPU relaxation to 1e-6 with all atoms free, and initial failures included in request accounting. Separating stages changes scheduling and timing accounting. Hardware/library versions are retained; mixed environments are not promised to be bitwise identical.


In [ ]:
PHASE = "prepare"  # prepare -> sample -> relax; rerun a paused phase unchanged
DRIVE_FOLDER = "IDL-PROJECT/gap2_colab_1024_standard_fast"
GPU_SESSION_HOURS = 4.0
CPU_SESSION_HOURS = 10.5
COMPUTE_UNITS_PER_GPU_HOUR = None  # optional: current rate shown by Colab Resources
AUTO_DISCONNECT_GPU = True
ALLOW_GPU_FOR_CPU_PHASE = False  # enabling this can consume units during CPU-only work

CONDITIONS = ["ka2d_1024_standard", "ka2d_1024_fast"]
MODELS = ["seed0", "seed1", "seed2", "sham_seed0"]
assert PHASE in {"prepare", "sample", "relax"}
assert GPU_SESSION_HOURS > 0 and CPU_SESSION_HOURS > 0
assert COMPUTE_UNITS_PER_GPU_HOUR is None or COMPUTE_UNITS_PER_GPU_HOUR > 0
print("Phase:", PHASE, "| Conditions:", CONDITIONS)


In [ ]:
import os, sys, subprocess, time, json, shutil, signal, tempfile, zipfile, hashlib
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
from google.colab import drive, userdata

if PHASE == "sample" and not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime for the sample phase")
if PHASE != "sample" and torch.cuda.is_available() and not ALLOW_GPU_FOR_CPU_PHASE:
    raise RuntimeError("Switch the accelerator to None for this CPU phase to conserve compute units")
drive.mount("/content/drive")
PERSIST = Path("/content/drive/MyDrive")/DRIVE_FOLDER
PERSIST.mkdir(parents=True, exist_ok=True)
ROOT = Path("/content/IDL-PROJECT-gap2-colab-code")
PUSHROOT = Path("/content/IDL-PROJECT-gap2-colab-publisher")
WORK = Path("/content/gap2-colab-work")
DATA = WORK/"study"
REPO = "https://github.com/chilalaabubakar/IDL-PROJECT.git"
TOKEN = userdata.get("GITHUB_TOKEN")
assert TOKEN, "Add and enable the Colab GITHUB_TOKEN secret"

def run(args, cwd=None, env=None):
    result=subprocess.run(list(map(str,args)),cwd=cwd,env=env,text=True,capture_output=True)
    if result.returncode:
        raise RuntimeError(result.stderr.replace(TOKEN,"[REDACTED]")[-4000:])
    return result.stdout

def atomic_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix(".tmp")
    temp.write_text(json.dumps(value,indent=2)+"\n")
    temp.replace(path)

def validate_saved_file(path):
    if path.suffix==".npz" or path.name.endswith(".npz.tmp"):
        with np.load(path,allow_pickle=False) as data:
            for key in data.files:
                data[key]  # decompression + CRC validation
    elif path.suffix==".json" or path.name.endswith(".json.tmp"):
        json.loads(path.read_text())

def copy_valid_tree(source, destination, keep_existing=False):
    source=Path(source);destination=Path(destination)
    if not source.exists():
        return
    if source.resolve()==destination.resolve() or destination.resolve().is_relative_to(source.resolve()):
        raise ValueError("Backup source and destination must be separate")
    for src in source.rglob("*"):
        if not src.is_file() or src.suffix not in {".npz",".json",".csv",".md",".log",".png",".pdf"}:
            continue
        dest=destination/src.relative_to(source)
        if keep_existing and dest.exists():
            try:
                validate_saved_file(dest)
                continue
            except (ValueError,OSError,EOFError,zipfile.BadZipFile):
                pass
        try:
            validate_saved_file(src)
        except (ValueError,OSError,EOFError,zipfile.BadZipFile):
            print("Skipping incomplete/corrupt file:",src.name)
            continue
        dest.parent.mkdir(parents=True,exist_ok=True)
        temp=dest.with_suffix(dest.suffix+".tmp")
        shutil.copy2(src,temp)
        validate_saved_file(temp)
        temp.replace(dest)

if not ROOT.exists():
    run(["git","clone",REPO,ROOT])
else:
    assert not run(["git","status","--porcelain"],ROOT).strip(), "Source checkout has local edits"
    run(["git","fetch","origin","main"],ROOT)
if not PUSHROOT.exists():
    run(["git","clone",REPO,PUSHROOT])
else:
    assert not run(["git","status","--porcelain"],PUSHROOT).strip(), "Publisher has local edits"
    run(["git","fetch","origin","main"],PUSHROOT)
    run(["git","merge","--no-edit","origin/main"],PUSHROOT)

# Prefer Drive. If empty, restore the latest matching published Colab run.
settings_file=PERSIST/"run_config.json"
if not settings_file.exists():
    prior=sorted((PUSHROOT/"results").glob("colab_gap2_1024_standard_fast_*"))
    matching=[p for p in prior if (p/"run_config.json").exists()
              and json.loads((p/"run_config.json").read_text()).get("conditions")==CONDITIONS]
    if matching:
        copy_valid_tree(matching[-1],PERSIST)
if settings_file.exists():
    settings=json.loads(settings_file.read_text())
    assert settings["conditions"]==CONDITIONS and settings["batch_size"]==4, "Stored task settings differ"
    CODE_SHA=settings["code_revision"]
else:
    CODE_SHA=run(["git","rev-parse","origin/main"],ROOT).strip()
    settings=dict(code_revision=CODE_SHA,conditions=CONDITIONS,batch_size=4,
                  execution="separate-sampling-and-relaxation-v1",
                  run_name="colab_gap2_1024_standard_fast_"+datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%SZ"))
    atomic_json(settings_file,settings)
run(["git","checkout","--detach",CODE_SHA],ROOT)
run([sys.executable,"-m","pip","install","-q","-e",".[md]"],ROOT)

from importlib.metadata import distribution
mpi=distribution("mpich")
mpi_dirs=sorted({str(mpi.locate_file(f).resolve().parent) for f in mpi.files or []
                 if f.name=="libmpi.so.12" and mpi.locate_file(f).exists()})
assert mpi_dirs, "MPICH did not install libmpi.so.12"
ENV=dict(os.environ,OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
ENV["LD_LIBRARY_PATH"]=":".join(mpi_dirs+[str(Path(sys.prefix)/"lib"),ENV.get("LD_LIBRARY_PATH","")])
run([sys.executable,"-c","from lammps import lammps; x=lammps(cmdargs=['-screen','none','-log','none']); x.close()"],ROOT,ENV)
if (WORK/"run_config.json").exists():
    assert json.loads((WORK/"run_config.json").read_text())==settings, "Local results belong to another run"
WORK.mkdir(parents=True,exist_ok=True)
copy_valid_tree(PERSIST,WORK,keep_existing=True)
atomic_json(WORK/"run_config.json",settings)
PUBLISH=PUSHROOT/"results"/settings["run_name"]
PUBLISH.mkdir(parents=True,exist_ok=True)
START=time.monotonic()
LIMIT_HOURS=GPU_SESSION_HOURS if PHASE=="sample" else CPU_SESSION_HOURS
PHASE_STATUS="not_started"
ERROR=None
print("Code:",CODE_SHA,"| Drive:",PERSIST)
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0),"| Current CU/hour setting:",COMPUTE_UNITS_PER_GPU_HOUR)


In [ ]:
def completed_methods():
    return sum((DATA/c/m/"complete.json").exists() for c in CONDITIONS for m in MODELS+["hand_insert"])

def phase_completion():
    prepare=all((DATA/c/"hosts/protocol.json").exists() and (DATA/c/"requests.npz").exists() for c in CONDITIONS)
    sample=all((DATA/c/m/"generated/complete.json").exists() for c in CONDITIONS for m in MODELS)
    relax=completed_methods()==len(CONDITIONS)*5
    return dict(prepare=prepare,sample=sample,relax=relax)

def write_progress():
    done=phase_completion()
    combined=WORK/"combined/study_status.json"
    global_done=json.loads(combined.read_text()).get("complete",False) if combined.exists() else False
    value=dict(status="complete" if done["relax"] else "incomplete",phase=PHASE,phase_status=PHASE_STATUS,
               phases=done,task_id="colab_1024_standard_fast",assigned_conditions=CONDITIONS,
               task_complete=done["relax"],full_study_complete=global_done,code_revision=CODE_SHA,
               batch_size=4,n_hosts_per_condition=100,n_requests_per_model=256,
               session_hours=(time.monotonic()-START)/3600,torch=torch.__version__,cuda=torch.version.cuda,
               error=ERROR,updated_utc=datetime.now(timezone.utc).isoformat())
    atomic_json(WORK/"progress.json",value)

def backup():
    write_progress()
    copy_valid_tree(WORK,PERSIST)
    print("Drive checkpoint saved",datetime.now(timezone.utc).strftime("%H:%M UTC"),flush=True)

def push():
    copy_valid_tree(WORK,PUBLISH)
    run(["git","config","user.name","Gap 2 Colab evaluation"],PUSHROOT)
    run(["git","config","user.email","gap2-colab@users.noreply.github.com"],PUSHROOT)
    run(["git","add",str(PUBLISH.relative_to(PUSHROOT))],PUSHROOT)
    if not run(["git","diff","--cached","--name-only"],PUSHROOT).strip():
        return
    run(["git","commit","-m","Save Colab gap 2 checkpoints ("+PHASE+", "+PHASE_STATUS+")"],PUSHROOT)
    with tempfile.TemporaryDirectory() as td:
        helper=Path(td)/"askpass.py"
        helper.write_text("#!/usr/bin/env python3\nimport os,sys\nprint('x-access-token' if 'Username' in sys.argv[1] else os.environ['COLAB_GAP2_TOKEN'])\n")
        helper.chmod(0o700)
        auth=dict(os.environ,GIT_ASKPASS=str(helper),GIT_TERMINAL_PROMPT="0",COLAB_GAP2_TOKEN=TOKEN)
        remote=["git","-c","credential.helper="]
        for attempt in range(3):
            run(remote+["fetch","origin","main"],PUSHROOT,auth)
            run(["git","merge","--no-edit","origin/main"],PUSHROOT)
            result=subprocess.run(remote+["push","origin","HEAD:main"],cwd=PUSHROOT,env=auth,text=True,capture_output=True)
            if result.returncode==0:
                print("GitHub checkpoint saved:",settings["run_name"],flush=True)
                return
            detail=result.stderr.replace(TOKEN,"[REDACTED]")[-2000:]
            if any(x in detail.lower() for x in ["403","401","authentication failed","permission to"]):
                raise RuntimeError("Colab GITHUB_TOKEN needs repository Contents: Read and write.\n"+detail)
        raise RuntimeError("GitHub push failed; Drive backup remains available.\n"+detail)

def checkpoint(allow_publish_failure=False):
    backup()
    try:
        push()
    except RuntimeError as exc:
        if not allow_publish_failure:
            raise
        print("GitHub publishing failed; Drive checkpoint is saved:",str(exc),flush=True)

checkpoint()  # Verify write access before costly work. Drive remains the primary backup.
print("Published run:","https://github.com/chilalaabubakar/IDL-PROJECT/tree/main/results/"+settings["run_name"])


In [ ]:
def sampling_estimate():
    info=[]
    for f in DATA.glob("*/seed*/generated/batch_*.json"):
        info.append(json.loads(f.read_text()))
    # Include the sham's timings as well once they exist.
    for f in DATA.glob("*/sham_seed0/generated/batch_*.json"):
        info.append(json.loads(f.read_text()))
    measured_n=sum(x["n_requests"] for x in info)
    if not measured_n:
        return
    measured_seconds=sum(x["seconds"] for x in info)
    sampled=sum(len(list((DATA/c/m/"generated").glob("batch_*.npz")))*4 for c in CONDITIONS for m in MODELS)
    remaining=max(0,len(CONDITIONS)*len(MODELS)*256-sampled)
    hours=remaining*(measured_seconds/measured_n)/3600
    print(f"Sampled {sampled}/2048; rough GPU sampling time remaining {hours:.2f} h",flush=True)
    if COMPUTE_UNITS_PER_GPU_HOUR:
        print(f"At your entered rate: approximately {hours*COMPUTE_UNITS_PER_GPU_HOUR:.1f} more units for sampling, excluding setup/backup time",flush=True)
    else:
        print("Read the live Colab CU/hour rate to convert this estimate to units; it is not a billing guarantee",flush=True)

def monitor(jobs):
    processes=[];handles=[]
    last_save=time.monotonic();last_report=0
    try:
        for name,args in jobs:
            handle=(WORK/(name+".log")).open("a");handles.append(handle)
            processes.append(subprocess.Popen(args,cwd=ROOT,env=ENV,stdout=handle,stderr=subprocess.STDOUT,start_new_session=True))
        while any(p.poll() is None for p in processes):
            if any(p.poll() not in (None,0) for p in processes):
                raise RuntimeError("A stage failed; inspect the phase logs saved to Drive")
            if time.monotonic()-START>LIMIT_HOURS*3600:
                raise TimeoutError("Session budget reached. Rerun this phase to resume.")
            if time.monotonic()-last_report>60:
                print("Elapsed phase minutes:",round((time.monotonic()-START)/60),flush=True)
                for name,_ in jobs:
                    print(name,(WORK/(name+".log")).read_text()[-600:],flush=True)
                if PHASE=="sample":sampling_estimate()
                last_report=time.monotonic()
            if time.monotonic()-last_save>300:
                checkpoint(allow_publish_failure=True)
                last_save=time.monotonic()
            time.sleep(10)
        if any(p.returncode!=0 for p in processes):
            raise RuntimeError("A stage failed; inspect the saved logs")
    finally:
        for p in processes:
            if p.poll() is None:
                try:os.killpg(p.pid,signal.SIGTERM)
                except ProcessLookupError:pass
        for p in processes:
            if p.poll() is None:
                try:p.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    try:os.killpg(p.pid,signal.SIGKILL)
                    except ProcessLookupError:pass
                    p.wait()
        for h in handles:h.close()

def cmd(script,mode,condition,extra=()):
    return [sys.executable,"-u",script,mode,"--condition",condition,"--out",str(DATA)]+list(extra)

def collect_reports():
    kaggle=ROOT/"results/kaggle_gap2_kaggle_remaining_20261101-220126Z"
    return run([sys.executable,"scripts/collect_gap2.py","--source",str(kaggle),str(DATA),
                "--out",str(WORK/"combined")],ROOT,ENV)

try:
    PHASE_STATUS="running"
    if PHASE=="prepare":
        workers=min(2,os.cpu_count() or 1)
        for condition in CONDITIONS:
            monitor([(condition+"_hosts",cmd("scripts/run_gap2.py","hosts",condition,["--workers",str(workers)]))])
            monitor([(condition+"_requests",cmd("scripts/run_gap2.py","requests",condition))])
            checkpoint(allow_publish_failure=True)
    elif PHASE=="sample":
        # First batch provides a measured cost estimate; the full call reuses it.
        first=CONDITIONS[0]
        monitor([("sampling_benchmark",cmd("scripts/run_gap2_colab.py","sample",first,
                 ["--model","seed0","--device","cuda","--max-batches","1"]))])
        sampling_estimate()
        checkpoint(allow_publish_failure=True)
        for condition in CONDITIONS:
            for model in MODELS:
                monitor([(condition+"_"+model+"_sample",cmd("scripts/run_gap2_colab.py","sample",condition,
                         ["--model",model,"--device","cuda"]))])
                checkpoint(allow_publish_failure=True)
    else:
        jobs=[]
        for condition in CONDITIONS:
            for model in MODELS:
                if list((DATA/condition/model/"generated").glob("batch_*.npz")):
                    jobs.append((condition+"_"+model+"_relax",cmd("scripts/run_gap2_colab.py","relax",condition,["--model",model])))
            jobs.append((condition+"_insertion",cmd("scripts/run_gap2.py","insertion",condition,["--batch-size","4"])))
        workers=min(2,os.cpu_count() or 1)
        for start in range(0,len(jobs),workers):
            monitor(jobs[start:start+workers])
            checkpoint(allow_publish_failure=True)
        print(collect_reports())
    PHASE_STATUS="complete" if phase_completion()[PHASE] else "partial"
except TimeoutError as exc:
    PHASE_STATUS="paused";ERROR=str(exc);print(ERROR,flush=True)
except BaseException as exc:
    PHASE_STATUS="failed";ERROR=(type(exc).__name__+": "+str(exc)).replace(TOKEN,"[REDACTED]")
    raise
finally:
    if PHASE=="relax":
        try:print(collect_reports())
        except Exception as exc:print("Collection is pending:",str(exc).replace(TOKEN,"[REDACTED]")[-1000:])
    checkpoint(allow_publish_failure=True)
    print("Phase status:",PHASE_STATUS,"| Completion:",phase_completion(),flush=True)
    if PHASE=="sample" and AUTO_DISCONNECT_GPU:
        print("GPU checkpoints saved. Disconnecting; use CPU for the relax phase.",flush=True)
        from google.colab import runtime
        try:
            runtime.unassign()
        except Exception:
            print("Automatic disconnect is unavailable; disconnect the runtime manually to stop GPU allocation.")


## Read the result after the CPU relaxation phase

`progress.json` in Drive records preparation/sampling/relaxation status. `task_complete=true` covers the two Colab conditions. `combined/study_status.json: complete=true` means the four completed Kaggle conditions and both Colab conditions passed collection and the six-condition placement study is complete. Partial phases remain resumable; do not equate GPU sampling completion with finished relaxed placements.

Final tables: `combined/placement_summary.csv` and `combined/paired_contrasts.csv`. The primary denominator excludes sites already D2. Bootstrap intervals resample whole hosts, and the three-seed mean conditions on those three fixed models. The seed0 sham is a same-checkpoint control for seed0; other seeds also change weights. Placement transfer does not establish natural structural equivalence or finite-temperature transfer.


In [ ]:
if PHASE=="relax" and (WORK/"combined/placement_summary.csv").exists():
    import pandas as pd
    summary=pd.read_csv(WORK/"combined/placement_summary.csv")
    display(summary[summary.definition=="initially_nonD2"])
    print(json.loads((WORK/"combined/study_status.json").read_text()))
print("Drive checkpoint folder:",PERSIST)
